# 🚀 Week 4-5: Anomaly Detection Training

## Experiments Overview
- **Group A**: PatchCore, SoftPatch, EfficientAD (12 experiments)
- **Group B**: Ablation Study (9 experiments)
- **Total**: 21 experiments × 15 categories = 315 runs

## Setup Requirements
1. GPU: P100 or T4 (enable in Settings)
2. Data: Add `mvtec-noisy-anomaly-detection` dataset
3. Code: Add `anomaly-detection-code` dataset OR clone from GitHub

## 📦 Cell 1: Install Dependencies

In [ ]:
# Install anomalib
!pip install -q anomalib

# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 📂 Cell 2: Setup Paths

In [ ]:
import sys
from pathlib import Path

# Option 1: If you uploaded code as Kaggle Dataset
CODE_PATH = '/kaggle/input/anomaly-detection-code/'
sys.path.append(CODE_PATH)

# Option 2: If you cloned from GitHub
# !git clone https://github.com/your-username/your-repo.git
# CODE_PATH = '/kaggle/working/your-repo/'
# sys.path.append(CODE_PATH)

# Data path
DATA_PATH = '/kaggle/input/mvtec-noisy-anomaly-detection/'

# Output path (Kaggle saves /kaggle/working/ automatically)
OUTPUT_PATH = '/kaggle/working/results/'
Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)

print(f"Code path: {CODE_PATH}")
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")

# Verify data
!ls {DATA_PATH}

## 🔧 Cell 3: Update Config for Kaggle

In [ ]:
import yaml
import shutil

# Copy config to working directory
config_src = f'{CODE_PATH}/config/model_config.yaml'
config_dst = '/kaggle/working/model_config.yaml'
shutil.copy(config_src, config_dst)

# Update paths for Kaggle
with open(config_dst, 'r') as f:
    config = yaml.safe_load(f)

# Update output directories
config['logging']['results_dir'] = OUTPUT_PATH
config['logging']['checkpoints_dir'] = '/kaggle/working/checkpoints/'
config['logging']['logs_dir'] = '/kaggle/working/logs/'

# Save updated config
with open(config_dst, 'w') as f:
    yaml.dump(config, f)

print("✅ Config updated for Kaggle")
print(f"Results will be saved to: {OUTPUT_PATH}")

## 🧪 Cell 4: Test on 1 Category (IMPORTANT!)

In [ ]:
# Test PatchCore on bottle category only
# This ensures everything works before running all 315 experiments

import os
os.chdir('/kaggle/working/')

# Copy necessary files
!cp -r {CODE_PATH}/src .
!cp -r {CODE_PATH}/scripts .
!cp -r {CODE_PATH}/config .

# Update data path in code (if needed)
# You may need to modify src/data/dataset_loader.py to use DATA_PATH

# Run test
!python scripts/run_experiments.py --categories bottle --only patchcore

print("\n✅ Test completed! Check results:")
!ls -lh {OUTPUT_PATH}

## 🚀 Cell 5: Run ALL Experiments (Group A: PatchCore)

**Estimated time: ~1.5 hours**

In [ ]:
# Run PatchCore on all 15 categories, 4 variants
# Total: 60 experiments

!python scripts/run_experiments.py --only patchcore

print("\n✅ PatchCore completed!")
!ls -lh {OUTPUT_PATH}

## 🚀 Cell 6: Run ALL Experiments (Group A: SoftPatch)

**Estimated time: ~1 hour**

In [ ]:
# Run SoftPatch on all 15 categories, 4 variants
# Total: 60 experiments

!python scripts/run_experiments.py --only softpatch

print("\n✅ SoftPatch completed!")
!ls -lh {OUTPUT_PATH}

## 🚀 Cell 7: Run ALL Experiments (Group A: EfficientAD)

**Estimated time: ~8 hours** ⚠️ LONGEST

In [ ]:
# Run EfficientAD on all 15 categories, 4 variants
# Total: 60 experiments
# WARNING: This takes the longest time!

!python scripts/run_experiments.py --only efficientad

print("\n✅ EfficientAD completed!")
!ls -lh {OUTPUT_PATH}

## 🚀 Cell 8: Run ALL Experiments (Group B: Ablation)

**Estimated time: ~2 hours**

In [ ]:
# Run Ablation Study: SoftPatch with LOF/Gaussian/KNN
# Total: 135 experiments (9 experiments × 15 categories)

!python scripts/run_experiments.py --only ablation

print("\n✅ Ablation Study completed!")
!ls -lh {OUTPUT_PATH}

## 📊 Cell 9: View Results Summary

In [ ]:
import pandas as pd
import glob

# Find latest results file
results_files = glob.glob(f'{OUTPUT_PATH}/all_results_*.csv')
if results_files:
    latest_results = max(results_files, key=os.path.getctime)
    print(f"Latest results: {latest_results}")
    
    # Load and display
    df = pd.read_csv(latest_results)
    print(f"\nTotal experiments: {len(df)}")
    print(f"\nColumns: {df.columns.tolist()}")
    
    # Summary by model and variant
    if 'image_AUROC' in df.columns:
        summary = df.groupby(['model', 'variant'])['image_AUROC'].agg(['mean', 'std', 'count'])
        print("\n📊 Image AUROC Summary:")
        print(summary)
        
        # Best model per variant
        print("\n🏆 Best Model per Variant:")
        best = df.loc[df.groupby('variant')['image_AUROC'].idxmax()]
        print(best[['variant', 'model', 'category', 'image_AUROC']])
else:
    print("⚠️ No results file found")

## 📈 Cell 10: Quick Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if results_files:
    df = pd.read_csv(latest_results)
    
    # Plot AUROC by model and variant
    plt.figure(figsize=(12, 6))
    
    for model in df['model'].unique():
        model_data = df[df['model'] == model]
        variant_means = model_data.groupby('variant')['image_AUROC'].mean()
        plt.plot(variant_means.index, variant_means.values, marker='o', label=model, linewidth=2)
    
    plt.xlabel('Noise Variant', fontsize=12)
    plt.ylabel('Image AUROC', fontsize=12)
    plt.title('Model Performance vs Noise Level', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/quick_comparison.png', dpi=150)
    plt.show()
    
    print(f"\n✅ Figure saved to: {OUTPUT_PATH}/quick_comparison.png")

## 💾 Cell 11: Prepare Download

All files in `/kaggle/working/` will be saved to Output tab automatically.

**To download:**
1. Click **Save Version** (top right)
2. Choose **Save & Run All** or **Quick Save**
3. Wait for completion
4. Go to **Output** tab
5. Download files

In [ ]:
# List all output files
print("📦 Files ready for download:\n")
!ls -lhR /kaggle/working/results/

print("\n✅ All experiments completed!")
print("\n📥 Next steps:")
print("1. Click 'Save Version' (top right)")
print("2. Go to 'Output' tab")
print("3. Download results/ folder")
print("4. Copy to your local project")
print("5. Run analysis locally")

## 🎉 DONE!

### What you have now:
- ✅ 315 experiments completed
- ✅ Results CSV/JSON files
- ✅ Quick comparison figure
- ✅ Ready for analysis

### Next: Week 6-7 Analysis (on local machine)
```bash
python scripts/run_experiments.py --analyze --results results/all_results_YYYYMMDD.csv
```